# Light vs. dark: calcium imaging analysis

This notebook analyses calcium imaging recordings from cortical layers 4 and 6 (folders `L4` and
`L6`) while the light alternates between on and off. Each recording is a `.bob2` file made with
[richard](https://pypi.org/project/bbo-richard/): the raw movie, the scan and motion correction,
the regions of interest (ROIs) with their neuron IDs, and the ROI means. Rejected frames are
marked in the file.

The notebook

1. loads the recordings of one dataset and concatenates their ΔF/F₀ traces,
2. shows all neurons as a heatmap together with the light state,
3. deconvolves the traces with [OASIS](https://github.com/j-friedrich/OASIS) into denoised calcium
   traces and inferred spike trains,
4. compares activity while the light is on with activity while it is off.

## Setup

Run this cell once per Colab session. It

* installs richard (reads `.bob2` files), ipympl (interactive figures), tqdm (progress bars) and
  OASIS (`oasis-deconv`, deconvolution),
* downloads the course's helper code from GitHub and makes it importable as `helper`,
* downloads the light–dark data from Nextcloud into `data/lightdark`,
* switches matplotlib to interactive figures (`%matplotlib widget`).

Colab loads matplotlib before ipympl is installed, and matplotlib only looks for extra plotting
backends once. `backend_registry._clear()` makes it look again, so `widget` is found without
restarting the session.

In [ ]:
!pip install -q bbo-richard ipympl tqdm oasis-deconv

# Download helper code
import os, sys, subprocess

REPO_URL = "https://github.com/bbo-lab/TimeSeriesAnalysis_CaImaging_2026"
REPO_DIR = "/content/TimeSeriesAnalysis_CaImaging_2026"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Download data
from helper.nextcloud_data import get_lightdark_data
SHARE_URL = "https://cloud.bbo.mpinb.mpg.de/s/3pMzwrPq6QbKoxo"
datasets = get_lightdark_data(SHARE_URL)

# Set up notebook for ipympl
from google.colab import output
output.enable_custom_widget_manager()

from matplotlib.backends import backend_registry
# Colab loads matplotlib before ipympl is installed, and matplotlib only looks for 
# extra plotting backends once. backend_registry._clear() makes it look again, 
# so widget is found without restarting the session.
backend_registry._clear()
%matplotlib widget

## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

## Choose a dataset

A dataset is one field of view; its name ends with the cortical layer, e.g. `fs2g010_2_L6`. Each
`.bob2` file is one recording of that field of view and is named after the dataset and the original
recording. Neurons recorded in several files of the same dataset keep their neuron ID.

The light periods are stored per experiment in `metadata/<experiment>_light_on.txt` and
`..._light_off.txt`: one period per row with start and end time in seconds, on the same clock as
the frame times of the recordings. `light["on"]` and `light["off"]` hold them as pandas data frames
with the columns `start` and `end`. Times that are in neither are undefined.

Where one recording ends and the next begins, the files split a period into two, about 2 s apart,
although the light did not change. `merge_split_periods` joins these again, so every light period
is counted once in the analysis. Longer gaps (14 s and more in these files) are real pauses with an
unknown light state and are kept.

All recordings of the dataset are used (`f"{DATASET}_*.bob2"`) and concatenated in the next
step. To analyse a single recording, put its full file name into the pattern.

In [ ]:
for d in datasets:
    print(d)

In [ ]:
DATASET = "fs2g010_2_L6"

MAX_SPLIT = 5.0   # s; a shorter gap between two periods of the same state is a break between recordings


def merge_split_periods(periods, max_gap=MAX_SPLIT):
    """Join consecutive periods of one light state that are less than max_gap seconds apart.

    The light files split a period where one recording ends and the next starts (gaps of about 2 s),
    although the light stayed the same. Longer gaps are real pauses and are kept. Two periods of the
    same state with an opposite period in between are always far more than max_gap apart.
    """
    periods = periods.sort_values("start", ignore_index=True)
    new_period = periods["start"] - periods["end"].shift() > max_gap   # True where a separate period starts
    return periods.groupby(new_period.cumsum()).agg(start=("start", "min"), end=("end", "max")).reset_index(drop=True)


# Light periods of the experiment: one row per period with start and end time (s).
# The columns are separated by tabs or spaces, depending on the file.
light_status = ("on", "off")
light = { ls: merge_split_periods(pd.read_csv(f"data/lightdark/metadata/{DATASET.split("_")[0]}_light_{ls}.txt",
                                              sep=r"\s+", header=None, names=["start", "end"]))
          for ls in light_status }

# Recordings of the dataset; the layer folder (L4 or L6) is the last part of the dataset name.
# glob returns the files in arbitrary order; sorted by name they are in temporal order.
bob2_files = sorted(Path(f"data/lightdark/{DATASET.split("_")[-1]}/").glob(f"{DATASET}_*.bob2"))

bob2_files

## Load ΔF/F₀

richard stores every processing step as a filter of a channel. We use channel 1 after scan
correction and motion correction, and read

* `roi_means`: the mean fluorescence of every ROI in every frame,
* `dff_values`: ΔF/F₀ in percent, `(F − F₀) / F₀ · 100`. F₀ is the mean of the lower half of the
  slightly smoothed trace. With a window at least twice as long as the recording, as here, there is
  one F₀ for the whole recording, so slow changes within the recording (e.g. between light periods) are kept. A
  shorter window gives a moving F₀ that follows, and removes, such slow changes.

Both are arrays of frames × neurons; `neuron_ids` gives the neuron of every column and `times` the
frame times in seconds. Rejected frames are left out of F₀ and are NaN in ΔF/F₀.

With several files, every file is read on its own into a table of frame times × neuron IDs, and
`pd.concat` stacks the tables in time. Because the columns are labelled with neuron IDs, pandas
matches them between files; a neuron without an ROI in one of the files is NaN there. The files of
a dataset often contain different neurons, so stacking plain numpy arrays would mix up columns.

`dff_table` and `roi_table` keep the labels (`dff_table.loc[1]` is the second recording,
`dff_table[neuron_id]` one neuron); `dff_values` and `roi_means` are the same data as numpy arrays.
`recording` gives the file index of every frame. F₀ is computed per file, so the same neuron can
have a different F₀ in each file.

In [ ]:
from richard import Workspace, DffSettings

# The settings cannot be changed after creation; pass them here (or use dataclasses.replace)
dff_settings = DffSettings(
    window=2000000.0,        # baseline window; at least twice the recording gives one F₀ for the whole recording
    baseline_sigma=3.0,      # Gaussian smoothing (σ) of the trace before F₀ is computed, against noise
    percentile=50.0,         # F₀ = mean of the smoothed values below this percentile (within the window)
    offset=0.0,              # subtracted from the fluorescence first, e.g. a background or dark level
    units='samples',         # unit of window and baseline_sigma: 'samples' (frames) or 'seconds'
)

dff_tables, mean_tables = [], []                  # one table per file: frame times × neuron IDs
for path in bob2_files:
    with Workspace.open(path) as ws:
        # Channel 1 after scan correction followed by motion correction
        sel = ws.select(channel=1, filter=["scan_correction", "motion_correction"])

        means = sel.roi_means()                   # stored ROI averages
        dff = sel.dff(settings=dff_settings)      # ΔF/F₀ in percent, computed in memory; F₀ per file

    dff_tables.append(pd.DataFrame(dff.values, index=means.times, columns=means.neuron_ids))
    mean_tables.append(pd.DataFrame(means.values, index=means.times, columns=means.neuron_ids))

# Stack in time; columns are matched by neuron ID, NaN where a file has no ROI for a neuron
dff_table = pd.concat(dff_tables, keys=range(len(dff_tables)), names=["recording", "time"])
roi_table = pd.concat(mean_tables, keys=range(len(mean_tables)), names=["recording", "time"])
assert dff_table.index.get_level_values("time").is_monotonic_increasing, "bob2_files are not in temporal order"

roi_means = roi_table.to_numpy()                  # (t, n) frames × neurons, all recordings
dff_values = dff_table.to_numpy()                 # (t, n) frames × neurons; NaN at rejected frames
neuron_ids = list(dff_table.columns)              # column order of both arrays
times = dff_table.index.get_level_values("time").to_numpy()            # (t,) frame times in seconds
recording = dff_table.index.get_level_values("recording").to_numpy()   # (t,) file index of every frame
frame_interval = np.median(np.diff(times))        # seconds per frame

print(f"{len(bob2_files)} recordings, {len(times)} frames, {len(neuron_ids)} neurons")

## Overview: ΔF/F₀ of all neurons

Each row of the heatmap is one neuron, each column one frame. White is ΔF/F₀ = 0, red is above and
blue below the baseline F₀; the colour range covers the 1st to 99th percentile of all values.
The bar above shows the light: warm yellow while it is on, dark blue while it is off, grey when the
state is undefined. Every recording is drawn as its own image, so the gaps between recordings stay
empty. Rejected frames are NaN and show as white gaps.

In [ ]:
low, high = np.nanpercentile(dff_values, [1, 99])   # colour range: 1st to 99th percentile
light_colors = {"on": "#f2b705", "off": "#1b2a4e"}  # warm yellow, dark blue; undefined stays grey

fig, (ax_light, ax) = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, layout='constrained',
                                   gridspec_kw=dict(height_ratios=[1, 24]))

# Light status: grey background, coloured spans where the light is on or off
ax_light.set_facecolor("#b0b0b0")
for ls, color in light_colors.items():
    for start, end in light[ls].itertuples(index=False):
        ax_light.axvspan(start, end, color=color, linewidth=0)
ax_light.set_yticks([])
ax_light.set_ylabel('light', rotation=0, ha='right', va='center')
# handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in [*light_colors.values(), "#b0b0b0"]]
# ax_light.legend(handles, ['on', 'off', 'undefined'], loc='lower right', bbox_to_anchor=(1, 1),
#                 ncols=3, frameon=False)

# ΔF/F₀: one image per recording, so the gaps between recordings stay empty
norm = TwoSlopeNorm(vmin=low, vcenter=0, vmax=high)  # white at ΔF/F₀ = 0
for r in np.unique(recording):
    rows = recording == r
    image = ax.imshow(dff_values[rows].T, aspect='auto', interpolation='nearest', cmap='RdBu_r', norm=norm,
                      extent=(times[rows][0] - frame_interval / 2, times[rows][-1] + frame_interval / 2,
                              len(neuron_ids) - 0.5, -0.5))
ax.set_xlim(times[0] - frame_interval / 2, times[-1] + frame_interval / 2)
ax.set_xlabel('time (s)')
ax.set_ylabel('neuron')
fig.colorbar(image, ax=[ax_light, ax], label='ΔF/F₀ (%)')
plt.show()

## OASIS on a single neuron

OASIS models a ΔF/F₀ trace as a calcium trace that decays exponentially between spikes, plus noise
and a constant baseline. `deconvolve` returns

* `c`: the denoised calcium trace,
* `s`: the inferred spike train (jumps of the calcium trace, in ΔF/F₀ units, not spike counts),
* `b`: the baseline,
* `g`: the decay per frame (`g = exp(−frame interval / decay time)`),
* `lam`: the sparsity penalty that was used.

The penalty decides how small events are treated: L1 (`penalty=1`) shrinks them to zero, L0
(`penalty=0`) gives a sparser, more spike-like output. The two plots compare them for one neuron.
The trace is NaN at rejected frames; see the note below the plot for what OASIS makes of that.

OASIS treats its input as one continuous trace. Across the gap between two recordings the calcium
trace would wrongly be continued, so this example uses only one recording. The analysis below
deconvolves every recording separately.

In [ ]:
from oasis.functions import deconvolve

NEURON = 0                                       # column of dff_values
# OASIS treats its input as one continuous trace, so we use one recording only:
# the first one in which this neuron has data
RECORDING = recording[np.isfinite(dff_values[:, NEURON])][0]
rows = recording == RECORDING                    # frames of that recording

y = dff_values[rows, NEURON] / 100               # one neuron, ΔF/F₀ as a fraction
recording_times = times[rows]                    # frame times of that recording (s)

# 0: L0 penalty - sparse, spike-like output
# 1: L1 penalty - small events are shrunk to zero
res_penalty = { penalty: deconvolve(y, penalty=penalty) for penalty in (0,1) }

fig, axs = plt.subplots(len(res_penalty),1, figsize=(12, 6.5), sharex=True)
for penalty, ax in zip(res_penalty,axs):
    # c: denoised calcium trace, s: inferred spike train, b: baseline,
    # g: AR coefficient(s) (decay), lam: sparsity penalty
    (c, s, b, g, lam) = res_penalty[penalty]

    ax.plot(recording_times, y, label="ΔF/F₀")
    ax.plot(recording_times, c, label="denoised calcium trace")
    ax.axhline(b, label="baseline")
    ax.plot(recording_times, s, label='inferred "spike" train')
    ax.set_title(f"Penalty L{penalty} - decay g = {np.atleast_1d(g)[0]:.3f} per frame")

axs[0].legend()
fig.suptitle(f"neuron {neuron_ids[NEURON]}, {bob2_files[RECORDING].name}")
axs[-1].set_xlabel('time (s)')
plt.show()

**Note: OASIS does not handle missing frames.** For the default neuron of `fs2g010_2_L6`, the L0
plot shows a negative "spike" of about −1.6 at 721 s, although spikes cannot be negative. The trace
has 21 rejected frames (NaN) at 729.8–731.7 s, and OASIS passes the NaNs into its internal
calculation. The NaN frames themselves stay NaN in the output, but the fit around them goes wrong:
here 9 s before the gap. The rest of the trace is affected too: with the NaNs OASIS finds 556 frames
with spikes, with the gap filled by interpolation 832. Without the NaNs, i.e. with the trace cut
before the gap or with the gap interpolated, there is no negative spike. The analysis below
therefore uses `helper.oasis_deconv`, which fills short gaps by interpolation and splits the trace
at longer gaps before calling OASIS.

## Light on vs. light off

We want to know whether neurons are more active while the light is on or while it is off.
OASIS turns each ΔF/F₀ trace into a denoised calcium trace `c` and an inferred spike train `s`.
Summing `s` over a light period measures how much activity started in that period, without the
slow calcium decay smearing activity from one period into the next.

The analysis has five steps, each guarding against one pitfall:

1. **One decay per neuron.** The summed spikes of a period scale with `1 − g` (the decay
   coefficient), so all periods of a neuron are deconvolved with the same `g`.
2. **Rates, not sums.** Periods differ in length, and rejected frames remove samples, so we divide
   by the valid time.
3. **Skip the switch.** The first seconds after each switch are left out; responses to the switch
   itself are a different question.
4. **Compare within neurons.** `s` is in ΔF/F₀ units, not spike counts, and the signal per spike
   differs between neurons. We compare on vs. off per neuron and test across neurons.
5. **Keep single periods.** One value per light period shows how consistent the effect is and
   reveals slow drifts.

Each caveat is shown on the data below.

In [ ]:
from helper.oasis_deconv import deconvolve_trace, deconvolve_traces


def neuron_decays(fits_per_neuron):
    """One decay coefficient g per neuron: the median over its stretches, weighted by stretch length."""
    decays = np.full(len(fits_per_neuron), np.nan)
    for n, fits in enumerate(fits_per_neuron):
        if not fits:
            continue
        g = np.array([fit["g"][0] for fit in fits])
        length = np.array([fit["stop"] - fit["start"] for fit in fits])
        order = np.argsort(g)
        cumulative = np.cumsum(length[order])
        decays[n] = g[order][np.searchsorted(cumulative, cumulative[-1] / 2)]
    return decays


def deconvolve_fixed_decay(dff, decays, penalty=1):
    """Deconvolve every neuron with its own fixed decay, so all its light periods share one g.

    Every recording is deconvolved separately: calcium does not carry over from one file to the next.
    """
    calcium = np.full(dff.shape, np.nan)
    spikes = np.full(dff.shape, np.nan)
    for n, g in enumerate(decays):
        if not np.isfinite(g):
            continue
        for r in np.unique(recording):
            rows = recording == r
            calcium[rows, n], spikes[rows, n], _ = deconvolve_trace(dff[rows, n], g=(g,), penalty=penalty)
    return calcium, spikes


# Pass 1: OASIS estimates the decay separately for every stretch of every neuron, recording by recording
free_fits = [[] for _ in neuron_ids]
for r in np.unique(recording):
    _, _, fits = deconvolve_traces(dff_values[recording == r])
    for n, neuron_fits in enumerate(fits):
        free_fits[n] += neuron_fits
decays = neuron_decays(free_fits)

# Pass 2: deconvolve again with one fixed decay per neuron
calcium, spikes = deconvolve_fixed_decay(dff_values, decays)

decay_times = -frame_interval / np.log(decays)
print(f"{np.isfinite(decays).sum()} of {len(decays)} neurons deconvolved; "
      f"decay time median {np.nanmedian(decay_times):.2f} s "
      f"(10–90%: {np.nanpercentile(decay_times, 10):.2f}–{np.nanpercentile(decay_times, 90):.2f} s)")

### Caveat 1: the decay sets the scale of the spike sums

OASIS defines a spike as the jump in calcium that the decay cannot explain, `s_t = c_t − g·c_{t−1}`.
Summed over a period this is roughly `(1 − g) · sum(c)`: the same calcium trace gives fewer spikes
with a slower decay. The cell below deconvolves one active neuron with the 10th, 50th and 90th
percentile of the fitted decays. The spike sum changes much more than the mean calcium.
Letting OASIS fit `g` per stretch would therefore mix decay differences into the light comparison.

In [ ]:
neuron = int(np.nanargmax(np.nanstd(dff_values, axis=0)))   # the most variable neuron
print(f"neuron {neuron_ids[neuron]}")
for g in np.nanpercentile(decays, [10, 50, 90]):
    c, s = deconvolve_fixed_decay(dff_values[:, [neuron]], [g])
    print(f"  g = {g:.3f} (τ = {-frame_interval / np.log(g):.2f} s):  "
          f"spike sum {np.nansum(s):7.0f}   (1 − g)·calcium sum {(1 - g) * np.nansum(c):7.0f}   "
          f"mean calcium {np.nanmean(c):5.1f} %")

# How much the free per-stretch fits of one neuron disagree
spread = [np.ptp([fit["g"][0] for fit in fits]) for fits in free_fits if len(fits) > 1]
print(f"range of g between stretches of the same neuron: median {np.median(spread):.3f}, max {np.max(spread):.3f}")

### Caveat 2: responses to the switch itself

Switching the light can cause a short response that is different from the sustained activity
during a period. The plot shows the population spike rate (mean over neurons) around every switch
in the recordings; switches closer than `WINDOW` to the start or end of a recording are left out.
The grey band is the time we leave out at the start of each period (`TRANSITION`). Adjust it if
the response lasts longer. With only a few switches per recording, these averages are noisy.

In [ ]:
TRANSITION = 3.0   # seconds left out at the start of every light period
WINDOW = 20.0      # seconds shown before and after each switch

population_rate = np.nanmean(spikes, axis=1) / frame_interval   # mean over neurons, per second
lags = np.arange(-round(WINDOW / frame_interval), round(WINDOW / frame_interval) + 1)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True, layout='constrained')
for ax, state in zip(axs, light_status):
    responses = []
    for switch in light[state]["start"]:            # start time of every period = a switch to this state
        i = np.searchsorted(times, switch)
        if i + lags[0] >= 0 and i + lags[-1] < len(times) and recording[i + lags[0]] == recording[i + lags[-1]]:
            # whole window inside one recording
            responses.append(population_rate[i + lags])
            ax.plot(lags * frame_interval, responses[-1], color='0.75', linewidth=0.8)
    if responses:
        ax.plot(lags * frame_interval, np.nanmean(responses, axis=0), color='k', label='mean')
    ax.axvspan(0, TRANSITION, color='0.85', zorder=0, label='left out')
    ax.axvline(0, color='k', linewidth=0.8)
    ax.set_title(f"light switched {state} (n = {len(responses)})")
    ax.set_xlabel('time from switch (s)')
axs[0].set_ylabel('population spike rate\n(ΔF/F₀ % per s)')
axs[0].legend(frameon=False)
plt.show()

### One value per neuron and light period

For every light period (minus the first `TRANSITION` seconds) and every neuron, we compute:

* `spike_rate`: summed spikes divided by the valid time (ΔF/F₀ % per second),
* `mean_dff`: the mean ΔF/F₀ as a cross-check that does not depend on deconvolution,
* `valid_s` and `coverage`: how much of the period has data.

Periods can be cut short by the start or end of the recordings, and rejected frames and image
edges remove samples. Periods with less than `MIN_VALID` seconds of data are skipped.

In [ ]:
MIN_VALID = 20.0   # seconds of valid data a period needs


def period_table(spikes):
    """One row per neuron and light period: spike rate, mean ΔF/F₀ and how much valid data it has."""
    rows = []
    for state in light_status:
        for start, end in light[state].itertuples(index=False):
            frames = (times >= start + TRANSITION) & (times < end)
            if not frames.any():
                continue                              # period outside the recordings
            for n, neuron_id in enumerate(neuron_ids):
                s = spikes[frames, n]
                valid = np.isfinite(s)
                valid_s = valid.sum() * frame_interval
                if valid_s < MIN_VALID:
                    continue
                rows.append(dict(neuron_id=neuron_id, state=state, start=start, end=end,
                                 valid_s=valid_s, coverage=valid.mean(),
                                 spike_rate=s[valid].sum() / valid_s,
                                 mean_dff=np.nanmean(dff_values[frames, n])))
    return pd.DataFrame(rows).sort_values(["start", "neuron_id"], ignore_index=True)


periods = period_table(spikes)

# How much data each period has (median over neurons); short periods are cut by the recordings
print(periods.groupby(["start", "end", "state"])[["valid_s", "coverage"]].median().round(2))

### Light on vs. off per neuron

Each neuron gets the mean over its on periods and over its off periods, so every period counts
equally. Neurons differ a lot in how large their signals are, so we compare them with
themselves: the Wilcoxon signed-rank test asks whether off − on is consistently positive or
negative across neurons. The modulation index `(off − on) / (off + on)` ranges from −1 (active
only with light on) to +1 (active only with light off).

In [ ]:
from scipy.stats import wilcoxon


def compare(periods, measure):
    """Per-neuron mean of a measure in on and off periods, and a paired test across neurons."""
    per_neuron = periods.pivot_table(index="neuron_id", columns="state", values=measure, aggfunc="mean").dropna()
    test = wilcoxon(per_neuron["off"], per_neuron["on"])
    higher = (per_neuron["off"] > per_neuron["on"]).sum()
    print(f"{measure}: {len(per_neuron)} neurons, median on {per_neuron['on'].median():.2f}, "
          f"off {per_neuron['off'].median():.2f}; higher with light off in {higher}; Wilcoxon p = {test.pvalue:.2g}")
    return per_neuron


rates = compare(periods, "spike_rate")
dffs = compare(periods, "mean_dff")
rates["modulation"] = (rates["off"] - rates["on"]) / (rates["off"] + rates["on"])

In [ ]:
fig, (ax_time, ax_pair, ax_index) = plt.subplots(1, 3, figsize=(14, 4), layout='constrained',
                                                 gridspec_kw=dict(width_ratios=[2, 1.2, 1]))

# Population median of every light period over time: is the effect consistent, is there a drift?
for (start, end, state), group in periods.groupby(["start", "end", "state"]):
    ax_time.hlines(group["spike_rate"].median(), max(start + TRANSITION, times[0]), min(end, times[-1]),
                   color=light_colors[state], linewidth=4)
ax_time.set_xlabel('time (s)')
ax_time.set_ylabel('median spike rate (ΔF/F₀ % per s)')
ax_time.set_title('light periods', loc='left')

# Every neuron on vs. off; log axes because neurons differ by orders of magnitude
positive = rates[(rates["on"] > 0) & (rates["off"] > 0)]
ax_pair.scatter(positive["on"], positive["off"], s=12, color='0.3')
limits = [positive[["on", "off"]].min().min() / 1.5, positive[["on", "off"]].max().max() * 1.5]
ax_pair.plot(limits, limits, color='0.6', linewidth=0.8)
ax_pair.set(xscale='log', yscale='log', xlim=limits, ylim=limits,
            xlabel='spike rate, light on', ylabel='spike rate, light off')
ax_pair.set_title(f'neurons ({len(rates) - len(positive)} with a zero rate not shown)', loc='left')

# Distribution of the modulation index
ax_index.hist(rates["modulation"].dropna(), bins=np.linspace(-1, 1, 21), color='0.5')
ax_index.axvline(0, color='k', linewidth=0.8)
ax_index.set_xlabel('modulation (off − on) / (off + on)')
ax_index.set_ylabel('neurons')
plt.show()

### Caveat 3: the sparsity penalty

The L1 penalty (`penalty=1`) shrinks small events to zero; L0 (`penalty=0`) keeps more of them.
The absolute rates therefore depend on the penalty. A real light effect should not: here the
modulation index is computed again with L0 and compared with L1.

In [ ]:
_, spikes_l0 = deconvolve_fixed_decay(dff_values, decays, penalty=0)
rates_l0 = compare(period_table(spikes_l0), "spike_rate")
rates_l0["modulation"] = (rates_l0["off"] - rates_l0["on"]) / (rates_l0["off"] + rates_l0["on"])

both = pd.concat([rates["modulation"], rates_l0["modulation"]], axis=1, keys=["L1", "L0"]).dropna()
print(f"modulation index L1 vs. L0: correlation {both.corr().iloc[0, 1]:.2f}, "
      f"same sign in {(np.sign(both['L1']) == np.sign(both['L0'])).mean():.0%} of {len(both)} neurons")

### Is it more than chance? A circular-shift test

The Wilcoxon test above treats neurons as independent, and it says nothing about single neurons.
To ask which neurons prefer light or dark, and whether that many such neurons could arise by
chance, we compare the real data with versions in which activity and light are no longer aligned.

**The test.** Within every recording, all spike trains are shifted together by a random number of
frames; frames pushed past the end of the recording wrap around to its start. The light periods
stay where they are. A shift keeps everything about the activity (calcium dynamics, slow drift,
correlations between neurons) but breaks its relation to the light. For every shift we compute the
modulation index of every neuron again, exactly as above. Repeated `N_SHIFTS` times, this gives
the distribution of modulation indices expected by chance.

* **Per neuron:** the p-value compares the real modulation index with the neuron's shifted ones:
  twice the fraction of shifts that lie beyond it on the nearer side. The shifted values need not
  be centred on 0 (e.g. when on and off periods have different lengths), so we look at the two
  tails separately instead of at the size of the index. Neurons with p < `ALPHA` count as
  **dark-sensitive** (more active with light off) or **light-sensitive** (more active with light on).
* **Number of sensitive neurons:** even without any light effect, about `ALPHA` of the neurons pass
  by chance, and because neurons fluctuate together, this number varies a lot between shifts. We
  therefore count the "sensitive" neurons in every shift as well. If the real count is larger than
  almost all shifted counts, it is not a result of chance. This also takes care of testing many
  neurons at once.
* **Population effect:** the same comparison for the median modulation index of all neurons
  replaces the Wilcoxon test, now with the correlations between neurons taken into account.

**Limitations.**

* Shifts shorter than `MIN_SHIFT` are not used, because the activity would still be almost aligned
  with the light.
* A recording is about 280 s long, a light cycle about 190 s. A shift by roughly one cycle aligns
  activity and light again, so some "chance" shifts still contain the real effect. This makes the
  test conservative: real effects are harder, not easier, to detect.
* The test is about one dataset, i.e. one field of view in one animal. A statement about a layer
  needs several datasets.

In [ ]:
N_SHIFTS = 1000   # number of random shifts
MIN_SHIFT = 30.0  # s; shorter shifts leave activity and light almost aligned
ALPHA = 0.05      # significance level for single neurons
rng = np.random.default_rng(0)   # fixed seed, so every run gives the same result

# Frames of every light period without its first TRANSITION seconds, as in period_table
period_masks, period_is_on = [], []
for state in light_status:
    for start, end in light[state].itertuples(index=False):
        frames = (times >= start + TRANSITION) & (times < end)
        if frames.any():
            period_masks.append(frames)
            period_is_on.append(state == "on")
period_masks = np.array(period_masks, dtype=float)   # (periods, t)
period_is_on = np.array(period_is_on)                # (periods,)


def modulation_index(spikes):
    """(off − on) / (off + on) of every neuron, computed like period_table and compare, but for all at once."""
    valid = np.isfinite(spikes)
    with np.errstate(divide="ignore", invalid="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)               # periods or neurons without data
        valid_s = period_masks @ valid * frame_interval               # (periods, neurons) seconds with data
        rate = period_masks @ np.where(valid, spikes, 0) / valid_s    # spike rate of every period
        rate[valid_s < MIN_VALID] = np.nan
        on = np.nanmean(rate[period_is_on], axis=0)
        off = np.nanmean(rate[~period_is_on], axis=0)
        return (off - on) / (off + on)


def shift_within_recordings(spikes, rng):
    """Roll the frames of every recording by a random number of frames, the same for all neurons."""
    shifted = np.empty_like(spikes)
    min_frames = round(MIN_SHIFT / frame_interval)
    for r in np.unique(recording):
        rows = np.flatnonzero(recording == r)
        shift = rng.integers(min_frames, len(rows) - min_frames + 1)
        shifted[rows] = np.roll(spikes[rows], shift, axis=0)
    return shifted


observed = modulation_index(spikes)                                   # (neurons,)
null = np.array([modulation_index(shift_within_recordings(spikes, rng)) for _ in range(N_SHIFTS)])   # (shifts, neurons)

# The vectorised version gives the same modulation index as period_table and compare above
assert np.allclose(pd.Series(observed, index=neuron_ids).loc[rates.index], rates["modulation"])

In [ ]:
def two_sided_p(value, chance):
    """Twice the fraction of chance values in the smaller tail beyond value (chance need not be centred on 0)."""
    chance = chance[np.isfinite(chance)]
    if np.isnan(value) or not len(chance):
        return np.nan
    upper = (1 + (chance >= value).sum()) / (1 + len(chance))
    lower = (1 + (chance <= value).sum()) / (1 + len(chance))
    return min(1.0, 2 * min(upper, lower))


# p-value of every neuron, compared with its own shifts
p_observed = np.array([two_sided_p(observed[n], null[:, n]) for n in range(len(neuron_ids))])

# The same for every shift, compared with the other shifts: how many neurons pass by chance?
p_null = np.full(null.shape, np.nan)
for n in range(null.shape[1]):
    finite = np.isfinite(null[:, n])
    others = np.sort(null[finite, n])
    upper = len(others) - np.searchsorted(others, null[finite, n], side="left")   # includes the shift itself
    lower = np.searchsorted(others, null[finite, n], side="right")
    p_null[finite, n] = np.minimum(1, 2 * np.minimum(upper, lower) / len(others))

sensitivity = pd.DataFrame(dict(modulation=observed, p=p_observed), index=neuron_ids)
sensitivity["class"] = np.select([(sensitivity["p"] < ALPHA) & (sensitivity["modulation"] > 0),
                                  (sensitivity["p"] < ALPHA) & (sensitivity["modulation"] < 0)],
                                 ["dark-sensitive", "light-sensitive"], "not sensitive")

counts = {   # observed count and the count in every shift
    "dark-sensitive": (((p_observed < ALPHA) & (observed > 0)).sum(), ((p_null < ALPHA) & (null > 0)).sum(axis=1)),
    "light-sensitive": (((p_observed < ALPHA) & (observed < 0)).sum(), ((p_null < ALPHA) & (null < 0)).sum(axis=1)),
}
median_observed = np.nanmedian(observed)
median_null = np.nanmedian(null, axis=1)
p_median = two_sided_p(median_observed, median_null)

tested = np.isfinite(observed).sum()
for name, (count, null_counts) in counts.items():
    p_count = (1 + (null_counts >= count).sum()) / (N_SHIFTS + 1)
    print(f"{name}: {count} of {tested} neurons; by chance {np.median(null_counts):.0f} "
          f"(95% of shifts ≤ {np.percentile(null_counts, 95):.0f}); p = {p_count:.3f}")
print(f"median modulation index {median_observed:.2f}; by chance 95% within "
      f"{np.percentile(median_null, 2.5):.2f} to {np.percentile(median_null, 97.5):.2f}; p = {p_median:.3f}")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.5), layout='constrained')
for ax, (name, (count, null_counts)), color in zip(axs, counts.items(), [light_colors["off"], light_colors["on"]]):
    ax.hist(null_counts, bins=np.arange(null_counts.max() + 2) - 0.5, color='0.7', label='shifted (chance)')
    ax.axvline(count, color=color, linewidth=2.5, label='observed')
    ax.set_xlabel(f'number of {name} neurons')
    ax.set_title(name, loc='left')
axs[0].set_ylabel('shifts')
axs[0].legend(frameon=False)

axs[2].hist(median_null, bins=40, color='0.7')
axs[2].axvline(median_observed, color='k', linewidth=2.5)
axs[2].set_xlabel('median modulation (off − on) / (off + on)')
axs[2].set_title('population', loc='left')
plt.show()

If the observed count lies far to the right of the grey distribution, there are more sensitive
neurons than chance can explain. `sensitivity` lists the modulation index, p-value and class of
every neuron. A single neuron with p < `ALPHA` can still be a false positive: with 100 neurons and
no effect at all, about 5 of them pass. Statements about single neurons are therefore weaker than
the count.

### Further caveats

* **F₀ per recording.** The ΔF/F₀ baseline F₀ is computed from each whole recording, so it depends on
  how much on and off time the recording contains. Compare on vs. off within a recording (or within
  a neuron) rather than absolute values between recordings.
* **Neurons are not independent.** They come from the same field of view and share slow
  fluctuations (e.g. arousal, brain state). The Wilcoxon p-value treats them as independent and is
  therefore too optimistic; the circular-shift test takes the shared fluctuations into account.
  The consistency across single periods (left plot) is a further guide.
* **Drift and after-effects.** If periods of the same state differ (e.g. the light-on period after a
  light-off period is more active than the one before), the activity may be drifting or the
  previous period may still have an effect. More periods and recordings are needed to tell.
* **Spike rates are not firing rates.** `s` is in ΔF/F₀ units. Without electrophysiology to
  calibrate it, only relative changes within a neuron are meaningful.
* **Agreement with mean ΔF/F₀.** If the spike rates and the plain mean ΔF/F₀ disagree, look at the
  traces before trusting either.